In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_colwidth', None)


In [ ]:
discriminatory_classed = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/discriminatory_subclasses/discriminatory_subclassified.csv")
discriminatory_classed.info()


In [ ]:
discriminatory_classed[['review_id','sub_categories','experiencer_is_target', 'parse_status','is_experienced', 'is_observed','multi_label']]

In [ ]:
discriminatory_classed[discriminatory_classed['sub_categories']=='[]'][['review_id','trans_review']].reset_index(drop=True)

In [ ]:
# what are the  nationalities of the sub_categories?
discriminatory_classed['sub_categories'].value_counts()

In [ ]:
perpetuated = discriminatory_classed[discriminatory_classed['sub_categories'].str.contains('perpetuated')]
experienced = discriminatory_classed[discriminatory_classed['sub_categories'].str.contains('experienced')]
observed = discriminatory_classed[discriminatory_classed['sub_categories'].str.contains('observed')]
none = discriminatory_classed[discriminatory_classed['sub_categories']=='[]']

In [ ]:
perpetuated['nationality'].value_counts()

In [ ]:
experienced['nationality'].value_counts().head(10)

In [ ]:
observed['nationality'].value_counts()

In [ ]:
perpetuated.shape, experienced.shape, observed.shape, none.shape

In [ ]:
subset_frames = {
    "perpetuated": perpetuated,
    "experienced": experienced,
    "observed": observed,
    "none": none,
}

percentage_tables = []

for sub_category, frame in subset_frames.items():
    clean_frame = (
    frame.dropna(subset=["nationality"])
    .assign(nationality=lambda df: df["nationality"].astype(str).str.strip())
    .loc[lambda df: df["nationality"] != ""]
    )

    total_subcategory = len(clean_frame)
    top_nationalities = (
        frame.dropna(subset=["nationality"])
        .assign(nationality=lambda df: df["nationality"].astype(str).str.strip())
        .loc[lambda df: df["nationality"] != ""]
        .groupby("nationality", as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values("count", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    top_nationalities["percentage"] = top_nationalities["count"] / total_subcategory * 100
    top_nationalities["sub_category"] = sub_category
    percentage_tables.append(top_nationalities[["sub_category", "nationality", "percentage"]])

nationality_percentage_table = (
    pd.concat(percentage_tables, ignore_index=True)
    .pivot(index="nationality", columns="sub_category", values="percentage")
    .fillna(0)
    .round(2)
    .reset_index()
)

nationality_percentage_table

### Social themes 

In [ ]:
ambiguous_reviews = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/ambiguous_reviews.csv")
ambiguous_reviews.info()

In [ ]:
lingusitic_pattern_data = pd.concat([discriminatory_classed, ambiguous_reviews], ignore_index=True)
lingusitic_pattern_data.info()

In [ ]:
lingusitic_pattern_data.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/lingusitic_pattern_data.csv", index=False)